In [1]:
import re
import numpy as np
import pandas as pd
import requests, sys
from BECancerResistome import beagle2vep

ENSEMBL VEP configurations


In [2]:
server = "https://rest.ensembl.org"
ext = "/vep/human/hgvs"
headers = {"Content-Type": "application/json", "Accept": "application/json"}

## Import Beagle sgRNA annotation


In [3]:
# Other files: data/BeagleCoelho/EG_collab-guides.txt
variants = pd.read_csv(f"data/GR-crisprbe-guides.txt", sep="\t")
variants.shape

(41298, 23)

In [5]:
variants["editor"] = (
    variants["Edit Type"].apply(lambda v: "ABE" if v == "A-G" else "CBE").values
)

In [6]:
variants.sample(10)

,Input,CRISPR Enzyme,Edit Type,Edit Window,Target Assembly,Target Genome Sequence,Target Gene ID,Target Gene Symbol,Target Gene Strand,Target Transcript ID,...,sgRNA Target Sequence Start Pos. (global),sgRNA Orientation,Nucleotide Edits (global),Guide Edits,Nucleotide Edits,Amino Acid Edits,Mutation Category,Constraint Violations,Note,editor
13097,ENST00000258148.11,SpyoCas9NG,C-T,4..8,GRCh38 (9606),NC_000012.12,ENSG00000135679,MDM2,+,ENST00000258148.11,...,68839453,antisense,NaN,NaN,NaN,NaN,NaN,poly(T):TTTTTTT,NaN,CBE
35191,ENST00000686064.1,SpyoCas9NG,C-T,4..8,GRCh38 (9606),NC_000017.11,ENSG00000170836,PPM1D,+,ENST00000686064.1,...,60663051,antisense,60663064G>A,C_7,975G>A,Ala326Thr,Missense,NaN,WARNING: Annotated CDS is 5' incomplete--base ...,CBE
6923,ENST00000539479.6,SpyoCas9NG,C-T,4..8,GRCh38 (9606),NC_000012.12,ENSG00000135679,MDM2,+,ENST00000539479.6,...,68809251,sense,"68809255C>T;68809256C>T, 68809258C>T","C_5_6, C_8","44C>T;45C>T, 47C>T","Thr15Ile, Thr16Ile","Missense, Missense",NaN,NaN,CBE
20514,ENST00000438015.7,SpyoCas9NG,A-G,4..8,GRCh38 (9606),NC_000011.10,ENSG00000149554,CHEK1,+,ENST00000438015.7,...,125653790,sense,"125653793A>G, 125653797A>G","A_4, A_8","1281A>G, 1285A>G","Lys427Lys, Asn429Asp","Silent, Missense",NaN,NaN,ABE
40725,ENST00000567897.6,SpyoCas9NG,C-T,4..8,GRCh38 (9606),NC_000016.10,ENSG00000166851,PLK1,+,ENST00000567897.6,...,23682053,antisense,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CBE
26079,ENST00000544373.5,SpyoCas9NG,C-T,4..8,GRCh38 (9606),NC_000011.10,ENSG00000149554,CHEK1,+,ENST00000544373.5,...,125633318,antisense,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CBE
12002,ENST00000350057.9,SpyoCas9NG,A-G,4..8,GRCh38 (9606),NC_000012.12,ENSG00000135679,MDM2,+,ENST00000350057.9,...,68839451,sense,68839455A>G,A_5,1007A>G,Asp336Gly,Missense,NaN,NaN,ABE
39908,ENST00000300093.9,SpyoCas9NG,A-G,4..8,GRCh38 (9606),NC_000016.10,ENSG00000166851,PLK1,+,ENST00000300093.9,...,23687606,sense,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ABE
24099,ENST00000428830.6,SpyoCas9NG,C-T,4..8,GRCh38 (9606),NC_000011.10,ENSG00000149554,CHEK1,+,ENST00000428830.6,...,125644504,sense,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CBE
30854,ENST00000305921.8,SpyoCas9NG,A-G,4..8,GRCh38 (9606),NC_000017.11,ENSG00000170836,PPM1D,+,ENST00000305921.8,...,60600508,antisense,NaN,NaN,NaN,NaN,NaN,poly(T):TTTT,NaN,ABE


## Format for ENSEMBL VEP input

Use the Beagle columns [input, Nucleotide Edits] and format as HGVS identifiers for ENSEMBL VEP. Examples include:

- `ENST00000618231.3:c.9G>C`
- `ENST00000471631.1:c.28_33delTCGCGG`
- `ENST00000285667.3:c.1047_1048insC`
- `5:g.140532G>C`


In [7]:
beagle2vep(variants.loc[7343])

['ENST00000539479.6:c.515C>T']

In [8]:
variants["hgvs"] = [
    "-" if r["Nucleotide Edits"] is np.nan else beagle2vep(r)
    for _, r in variants.iterrows()
]
variants.sample(10)

,Input,CRISPR Enzyme,Edit Type,Edit Window,Target Assembly,Target Genome Sequence,Target Gene ID,Target Gene Symbol,Target Gene Strand,Target Transcript ID,...,sgRNA Orientation,Nucleotide Edits (global),Guide Edits,Nucleotide Edits,Amino Acid Edits,Mutation Category,Constraint Violations,Note,editor,hgvs
30238,ENST00000369535.5,SpyoCas9NG,A-G,4..8,GRCh38 (9606),NC_000001.11,ENSG00000213281,NRAS,-,ENST00000369535.5,...,antisense,114708577A>G,A_7,528T>C,Asp176Asp,Silent,NaN,NaN,ABE,[ENST00000369535.5:c.528T>C]
32542,ENST00000305921.8,SpyoCas9NG,A-G,4..8,GRCh38 (9606),NC_000017.11,ENSG00000170836,PPM1D,+,ENST00000305921.8,...,antisense,60663473T>C,A_5,1739T>C,Met580Thr,Missense,NaN,NaN,ABE,[ENST00000305921.8:c.1739T>C]
19122,ENST00000368467.4,SpyoCas9NG,A-G,4..8,GRCh38 (9606),NC_000001.11,ENSG00000163344,PMVK,-,ENST00000368467.4,...,antisense,154929148A>G,A_8,188T>C,Leu63Pro,Missense,NaN,NaN,ABE,[ENST00000368467.4:c.188T>C]
19138,ENST00000368467.4,SpyoCas9NG,A-G,4..8,GRCh38 (9606),NC_000001.11,ENSG00000163344,PMVK,-,ENST00000368467.4,...,antisense,154929171A>G,A_7,165T>C,His55His,Silent,NaN,NaN,ABE,[ENST00000368467.4:c.165T>C]
8400,ENST00000393416.7,SpyoCas9NG,A-G,4..8,GRCh38 (9606),NC_000012.12,ENSG00000135679,MDM2,+,ENST00000393416.7,...,sense,"68816901A>G, 68816904A>G","A_4, A_7","339A>G, 342A>G","Leu113Leu, Gly114Gly","Silent, Silent",NaN,NaN,ABE,"[ENST00000393416.7:c.339A>G, ENST00000393416.7..."
24484,ENST00000427383.6,SpyoCas9NG,A-G,4..8,GRCh38 (9606),NC_000011.10,ENSG00000149554,CHEK1,+,ENST00000427383.6,...,antisense,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ABE,-
24629,ENST00000427383.6,SpyoCas9NG,C-T,4..8,GRCh38 (9606),NC_000011.10,ENSG00000149554,CHEK1,+,ENST00000427383.6,...,sense,NaN,NaN,NaN,NaN,NaN,poly(T):TTTT,NaN,CBE,-
28964,ENST00000532669.5,SpyoCas9NG,A-G,4..8,GRCh38 (9606),NC_000011.10,ENSG00000149554,CHEK1,+,ENST00000532669.5,...,sense,125635500A>G;125635501A>G,A_4_5,448A>G;449A>G,Asn150Gly,Missense,NaN,NaN,ABE,"[ENST00000532669.5:c.448A>G, ENST00000532669.5..."
23903,ENST00000428830.6,SpyoCas9NG,C-T,4..8,GRCh38 (9606),NC_000011.10,ENSG00000149554,CHEK1,+,ENST00000428830.6,...,sense,125643882C>T,C_4,905C>T,Pro302Leu,Missense,NaN,NaN,CBE,[ENST00000428830.6:c.905C>T]
1088,ENST00000450114.7,SpyoCas9NG,A-G,4..8,GRCh38 (9606),NC_000011.10,ENSG00000166483,WEE1,+,ENST00000450114.7,...,sense,"9576096A>G, 9576097A>G","A_5, A_6","782+3A>G, 782+4A>G","(NC), (NC)","Intron, Intron",NaN,NaN,ABE,"[ENST00000450114.7:c.782+3A>G, ENST00000450114..."


In [9]:
variants_hgvs = [v for vs in variants["hgvs"] if vs != "-" for v in vs]

Write variants HGVS input to file


In [10]:
len(variants_hgvs)

51110

In [13]:
with open("data/GR-crisprbe-guides-VEP-HGVS-input.txt", "w") as f:
    for variant in variants_hgvs:
        f.write(f"{variant}\n")

In [14]:
!head -n 10 data/GR-crisprbe-guides-VEP-HGVS-input.txt

ENST00000450114.7:c.-3G>A
ENST00000450114.7:c.-1G>A
ENST00000450114.7:c.-3G>A
ENST00000450114.7:c.-1G>A
ENST00000450114.7:c.2T>C
ENST00000450114.7:c.-3G>A
ENST00000450114.7:c.-1G>A
ENST00000450114.7:c.2T>C
ENST00000450114.7:c.-1G>A
ENST00000450114.7:c.3G>A


VEP command


In [13]:
# ./vep --af --af_1kg --af_gnomade --af_gnomadg --appris --biotype --buffer_size 500 --ccds --check_existing --distance 5000 --domains --gencode_primary --hgvs --mane --numbers --plugin OpenTargets,file=[path_to]/OTGenetics.tsv.gz --plugin CADD,snv=[path_to]/CADD_GRCh38_1.7_whole_genome_SNVs.tsv.gz,indels=[path_to]/CADD_GRCh38_1.7_InDels.tsv.gz --plugin REVEL,file=[path_to]/new_tabbed_revel_grch38.tsv.gz --plugin SpliceAI,snv=[path_to]/spliceai_scores.masked.snv.hg38.vcf.gz,indel=[path_to]/spliceai_scores.masked.indel.hg38.vcf.gz,snv_ensembl=[path_to]/spliceai_scores.raw.snv.ensembl_mane.grch38.110.vcf.gz --plugin Enformer,file=[path_to]/enformer_grch38.vcf.gz --plugin AlphaMissense,file=[path_to]/AlphaMissense_hg38.tsv.gz --plugin Blosum62 --plugin MaveDB,file=[path_to]/MaveDB_variants.tsv.gz --plugin ClinPred,file=[path_to]/ClinPred_hg38_sorted_tabbed.tsv.gz --plugin dbscSNV,[path_to]/dbscSNV1.1_GRCh38.txt.gz --plugin EVE,file=[path_to]/eve_merged.vcf.gz --plugin LOEUF,file=[path_to]/loeuf_dataset_grch38.tsv.gz,match_by=gene --plugin NMD --plugin AncestralAllele,[path_to]/homo_sapiens_ancestor_GRCh38_109.fa.gz --plugin MaxEntScan,[path_to]/maxentscan --plugin mutfunc,db=[path_to]/mutfunc_data.db,motif=1,int=1,mod=1,exp=1 --polyphen b --protein --pubmed --regulatory --show_ref_allele --sift b --species homo_sapiens --symbol --transcript_version --tsl --uniprot --uploaded_allele --var_synonyms --cache --input_file [input_data] --output_file [output_file]

In [14]:
!head -n 10 data/BeagleCoelho/EG_collab-guides-VEP-HGVS-output.txt

#Uploaded_variation	Location	Allele	Consequence	IMPACT	SYMBOL	Gene	Feature_type	Feature	BIOTYPE	EXON	INTRON	HGVSc	HGVSp	cDNA_position	CDS_position	Protein_position	Amino_acids	Codons	Existing_variation	REF_ALLELE	UPLOADED_ALLELE	DISTANCE	STRAND	FLAGS	SYMBOL_SOURCE	HGNC_ID	MANE	MANE_SELECT	MANE_PLUS_CLINICAL	TSL	APPRIS	CCDS	ENSP	SWISSPROT	TREMBL	UNIPARC	UNIPROT_ISOFORM	SIFT	PolyPhen	DOMAINS	HGVS_OFFSET	AF	AFR_AF	AMR_AF	EAS_AF	EUR_AF	SAS_AF	gnomADe_AF	gnomADe_AFR_AF	gnomADe_AMR_AF	gnomADe_ASJ_AF	gnomADe_EAS_AF	gnomADe_FIN_AF	gnomADe_MID_AF	gnomADe_NFE_AF	gnomADe_REMAINING_AF	gnomADe_SAS_AF	gnomADg_AF	gnomADg_AFR_AF	gnomADg_AMI_AF	gnomADg_AMR_AF	gnomADg_ASJ_AF	gnomADg_EAS_AF	gnomADg_FIN_AF	gnomADg_MID_AF	gnomADg_NFE_AF	gnomADg_REMAINING_AF	gnomADg_SAS_AF	CLIN_SIG	SOMATIC	PHENO	PUBMED	VAR_SYNONYMS	MOTIF_NAME	MOTIF_POS	HIGH_INF_POS	MOTIF_SCORE_CHANGE	TRANSCRIPTION_FACTORS	OpenTargets_geneId	OpenTargets_l2g	CADD_PHRED	CADD_RAW	REVEL	SpliceAI_pred_DP_AG	SpliceAI_pred_DP_AL	SpliceAI_pred_DP_DG

Export HGVS annotated Beagle file


In [15]:
variants.to_csv(f"data/BeagleCoelho/EG_collab-guides-HGVS.txt", sep="\t", index=False)

In [16]:
!head -n 10 data/BeagleCoelho/EG_collab-guides-HGVS.txt

Input	CRISPR Enzyme	Edit Type	Edit Window	Target Taxon	Target Assembly	Target Genome Sequence	Target Gene ID	Target Gene Symbol	Target Gene Strand	Target Transcript ID	Target Domain	sgRNA Sequence	sgRNA Context Sequence	PAM Sequence	sgRNA Sequence Start Pos. (global)	sgRNA Orientation	Nucleotide Edits (global)	Guide Edits	Nucleotide Edits	Amino Acid Edits	Mutation Category	Constraint Violations	Note	editor	hgvs
ENST00000262948.10	SpyoCas9NG	A-G	4..8	9606	GRCh38	NC_000019.10	ENSG00000126934	MAP2K2	-	ENST00000262948.10	CDS	CCGGGCTCCCTGCGTCCCGC	GTGGCCGGGCTCCCTGCGTCCCGCTGGTGA	TG	4090572	sense								ABE	-
ENST00000262948.10	SpyoCas9NG	C-T	4..8	9606	GRCh38	NC_000019.10	ENSG00000126934	MAP2K2	-	ENST00000262948.10	CDS	CCGGGCTCCCTGCGTCCCGC	GTGGCCGGGCTCCCTGCGTCCCGCTGGTGA	TG	4090572	sense	4090586G>A, 4090584G>A	C_6, C_8	*12C>T, *14C>T	(NC), (NC)	UTR, UTR			CBE	['ENST00000262948.10:c.*12C>T', 'ENST00000262948.10:c.*14C>T']
ENST00000262948.10	SpyoCas9NG	A-G	4..8	9606	GRCh38	NC_000019.10	ENSG000001

## [Optional] Rest API to query a single edit


In [17]:
vep_request = {"hgvs_notations": ["ENST00000429687.8:c.1553+4A>G"]}

In [18]:
# dict to string
vep_request = str(vep_request).replace("'", '"')
vep_request

'{"hgvs_notations": ["ENST00000429687.8:c.1553+4A>G"]}'

In [19]:
r = requests.post(
    server + ext,
    headers=headers,
    data=vep_request,
)

if not r.ok:
    r.raise_for_status()
    sys.exit()

decoded = r.json()
print(repr(decoded))

[{'colocated_variants': [{'strand': 1, 'id': 'rs1884202840', 'seq_region_name': '14', 'allele_string': 'A/G', 'end': 20357524, 'start': 20357524}], 'seq_region_name': '14', 'transcript_consequences': [{'strand': 1, 'biotype': 'protein_coding', 'impact': 'LOW', 'variant_allele': 'G', 'gene_symbol': 'PARP2', 'gene_id': 'ENSG00000129484', 'hgnc_id': 'HGNC:272', 'consequence_terms': ['splice_donor_region_variant', 'intron_variant'], 'transcript_id': 'ENST00000250416', 'gene_symbol_source': 'HGNC'}, {'hgnc_id': 'HGNC:272', 'gene_id': 'ENSG00000129484', 'consequence_terms': ['splice_donor_region_variant', 'intron_variant'], 'gene_symbol_source': 'HGNC', 'transcript_id': 'ENST00000429687', 'strand': 1, 'biotype': 'protein_coding', 'variant_allele': 'G', 'impact': 'LOW', 'gene_symbol': 'PARP2'}, {'strand': 1, 'distance': 1569, 'biotype': 'retained_intron', 'impact': 'MODIFIER', 'variant_allele': 'G', 'gene_symbol': 'PARP2', 'gene_id': 'ENSG00000129484', 'hgnc_id': 'HGNC:272', 'consequence_term

In [131]:
variant_vep = decoded[0]
variant_vep

{'strand': 1,
 'seq_region_name': '14',
 'end': 20357524,
 'transcript_consequences': [{'transcript_id': 'ENST00000250416',
   'gene_id': 'ENSG00000129484',
   'gene_symbol': 'PARP2',
   'variant_allele': 'G',
   'hgnc_id': 'HGNC:272',
   'gene_symbol_source': 'HGNC',
   'consequence_terms': ['splice_donor_region_variant', 'intron_variant'],
   'biotype': 'protein_coding',
   'strand': 1,
   'impact': 'LOW'},
  {'hgnc_id': 'HGNC:272',
   'gene_symbol_source': 'HGNC',
   'variant_allele': 'G',
   'impact': 'LOW',
   'consequence_terms': ['splice_donor_region_variant', 'intron_variant'],
   'strand': 1,
   'biotype': 'protein_coding',
   'gene_id': 'ENSG00000129484',
   'transcript_id': 'ENST00000429687',
   'gene_symbol': 'PARP2'},
  {'gene_id': 'ENSG00000129484',
   'transcript_id': 'ENST00000527384',
   'gene_symbol': 'PARP2',
   'distance': 1569,
   'hgnc_id': 'HGNC:272',
   'gene_symbol_source': 'HGNC',
   'variant_allele': 'G',
   'impact': 'MODIFIER',
   'consequence_terms': ['dow